# Lab 2 — Where Each Search Breaks (and Why You Need Both) — ES|QL edition

**Thesis:** Neither retriever wins everywhere. Semantic search **blurs exact identifiers** — error codes, config values, version strings — folding them into a region of vector space shared with everything conceptually nearby, so it can't *reliably* rank the one doc that matters. BM25 has its own failure modes: it can rank the **wrong exact match** (a boosted common-word title beating the rare token the user cared about), and it **buries** docs that share no vocabulary with a paraphrased query.

Seeing *when* and *why* each one fails is the prerequisite to building a hybrid retriever that covers both — Lab 3.

## What you'll learn
- Why semantic search **blurs** exact identifiers (and the honest cases where it nails them)
- Two ways BM25 fails: the **wrong exact match**, and **burying** paraphrased queries
- How to express BM25 in ES|QL with `MATCH(title, q, {"boost": 3.0}) OR MATCH(body, q)`
- ES|QL has **no `explain`** — so we read the score a different way: split the match into a title-only and a body-only query and compare. Arguably clearer than a nested JSON tree.

## ES|QL note: semantic vs BM25 is which field you MATCH
- `MATCH(body_semantic, q)` → **semantic** (it's a `semantic_text` field)
- `MATCH(title, q, {"boost": 3.0}) OR MATCH(body, q)` → **BM25** keyword search over text fields, with the title weighted 3× (the ES|QL equivalent of `multi_match` on `["title^3","body"]`).

> ⚠️ **Scoring differs from the retriever DSL.** `multi_match best_fields` takes the *max* of the per-field scores; the `OR`-composed `MATCH` here *sums* them. Absolute scores — and occasionally ranks — will differ from the retriever-DSL workshop. The *failure modes* are the same; exact numbers below are validated live (see `docs-esql/TRAP_QUERY_VALIDATION_ESQL.md`).

In [ ]:
# --- Workshop helpers (inline — same block across all ES|QL notebooks) ---
# ES|QL edition: every search runs through es.esql.query() instead of es.search().
# Defined inline so this notebook is self-contained and runs from the repo too.

import os, json, time
import requests
from elasticsearch import Elasticsearch

INDEX = "aiewf-workshop-docs"

ES_ENDPOINT = os.environ.get("ES_ENDPOINT")
ES_API_KEY  = os.environ.get("ES_API_KEY")
if not ES_ENDPOINT or not ES_API_KEY:
    raise ValueError(
        "Set ES_ENDPOINT and ES_API_KEY.\n"
        "  In Instruqt: pre-configured in the sandbox.\n"
        "  Re-running the repo: export ES_ENDPOINT=https://...  export ES_API_KEY=..."
    )

# request_timeout=120: RERANK and COMPLETION (Labs 4-5) call inference per row and
# can take several seconds — the default 10s would time out the LLM step.
es = Elasticsearch(ES_ENDPOINT, api_key=ES_API_KEY, request_timeout=120)

def esql(query, **params):
    """Run an ES|QL query with named parameters (?name in the query string).

    Usage:  esql(QUERY, q="securing cluster traffic")
    ES|QL named params take the form params=[{"name": value}, ...]. If your pinned
    client rejects named params, switch to positional `?` and params=[value, ...] —
    never f-string the query text in (injection + teaches the wrong pattern).
    """
    param_list = [{k: v} for k, v in params.items()] if params else None
    return es.esql.query(query=query, params=param_list, format="json")

def rows(resp):
    """Turn an ES|QL response ({columns, values}) into a list of dicts keyed by column."""
    cols = [c["name"] for c in resp["columns"]]
    return [dict(zip(cols, vals)) for vals in resp["values"]]

def show_esql(resp, fields=("id", "title", "summary"), score=True):
    """Pretty-print ES|QL rows as a ranked table (mirrors the DSL notebooks' show_hits)."""
    data = rows(resp)
    if not data:
        print("  (no rows)"); return
    for rank, r in enumerate(data, 1):
        cols = "  ".join(str(r.get(f, "")) for f in fields)
        sc = r.get("_score")
        s = f"  {sc:.4f}" if score and sc is not None else ""
        print(f"  #{rank:<2}{s}  {cols}")

print("✓ ES|QL helpers loaded")


# Lab 2 query templates and a side-by-side comparison helper.
Q_SEMANTIC = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body_semantic, ?q)
| SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score
"""

Q_BM25 = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(title, ?q, {"boost": 3.0}) OR MATCH(body, ?q)
| SORT _score DESC | LIMIT 5 | KEEP id, title, summary, _score
"""

def compare(query):
    """Run the same query through semantic and BM25 side by side (ES|QL)."""
    print(f"QUERY: {query!r}\n")
    print("SEMANTIC (vector):")
    show_esql(esql(Q_SEMANTIC, q=query))
    print("\nBM25 (lexical, title^3):")
    show_esql(esql(Q_BM25, q=query))

print("✓ Lab 2 helpers loaded")

In [ ]:
resp = esql("FROM aiewf-workshop-docs | STATS docs = COUNT(*)")
print(f"Connected to ES {es.info()['version']['number']} | {rows(resp)[0]['docs']} docs in '{INDEX}'")

## Failure mode 1: Exact identifiers — vector blurs the one token that matters

Semantic search is built to match *meaning*. That's exactly the wrong instinct for an **exact identifier** — an error code, a config value, a version string, a CVE number. The embedding model folds the identifier into a region of vector space shared with everything *conceptually nearby*, so the specific token stops working as a discriminator.

Our corpus has a JVM doc (`doc-007`) covering **OOMKilled / exit code 137**. We also seeded two distractor docs that talk about exit codes and process crashes *generically*, **without the literal "137"**. Read each hit's `summary` to see what it's actually about.

**Query:** `exit code 137`
- BM25 should pin `doc-007`: it contains the rare token `137`, high-IDF gold.
- Semantic will recognize the *concept* (a process being killed) — but can it tell `doc-007` apart from generic crash docs that mean almost the same thing?

In [ ]:
compare("exit code 137")

## What actually happened — semantic *blurs*, it doesn't cleanly miss

Look at the semantic column. `doc-007` is likely still **#1** — but by a hair, with distractor docs that never say "137" crammed right behind it. The model embedded "exit code 137" as the general concept "a process was killed," and in that neighborhood the OOM doc and the generic crash docs are nearly the **same point in vector space**. The `137` carries almost no weight.

That's more dangerous than a flat miss: the semantic *ranking is essentially noise*. Re-embed, change a chunk, add one similar doc, and #1 flips to a doc that doesn't even contain the number the user typed.

BM25, by contrast, wins decisively — the token `137` appears in exactly one doc, IDF rewards it, the ranking is stable.

---

### A cleaner miss: `new_primaries`

`exit code 137` was a *blur*. For a query semantic gets outright **wrong**, try a bare config value with no surrounding language. `new_primaries` is a value of the `cluster.routing.allocation.enable` setting — documented in `doc-008`.

In [ ]:
# Bare config value. Target: doc-008 (shard allocation settings).
# Semantic's #1 is a *plausible* doc (cluster health) — right neighborhood, wrong doc.
compare("new_primaries")

In [ ]:
# Honesty check: semantic is NOT always wrong on identifiers.
# A longer, distinctive dotted key carries enough structure to embed well.
# Here semantic should rank doc-008 #1. Don't overclaim the failure mode.
compare("cluster.routing.allocation.enable")

## Reading those two results together

- **`new_primaries`** → semantic puts a *cluster-health* doc at #1 and pushes the actual settings doc (`doc-008`) down. A bare token with no sentence around it, so the model guessed the topic and landed close but wrong.
- **`cluster.routing.allocation.enable`** → semantic gets it **right**. The dotted key is long and distinctive enough to embed into its own corner of vector space.

Exact identifiers aren't a guaranteed semantic failure — they're a **reliability** failure. Sometimes the model nails them, sometimes it blurs them, sometimes it picks a plausible neighbor. "Usually right" is not good enough when a user pastes an error code and needs *that* page. BM25's behavior on exact tokens is boring and predictable — exactly what you want for this query class.

## Failure mode 2: BM25 picks the *wrong* exact match

It's tempting to conclude "use BM25 for identifiers, semantic for everything else." But BM25 scores **lexical overlap**, weighted by field boosts and IDF — and that can reward a doc that shares the *common* words over the doc that shares the *rare, specific* one.

Three release-note-style docs:
- `doc-006` — title **"Elasticsearch breaking changes"** (generic)
- `doc-057` — title "Elasticsearch 8.18 release notes" (the one a user asking about 8.18 wants)
- `doc-058` — title "Elasticsearch 8.15 release notes"

**Query:** `8.18 breaking changes` — the user clearly wants the **8.18** page.

In [ ]:
compare("8.18 breaking changes")

## Reading the score WITHOUT `explain` — the ES|QL way

In the retriever-DSL workshop, this is where we'd run `explain=True` and read a nested JSON tree of `boost × idf × tf` factors to prove *why* `doc-006` beat `doc-057`. **ES|QL has no `explain` mode** — there's no per-document scoring tree.

That's fine, because we can show the same thing more directly: run the title-only match and the body-only match as **two separate queries** and compare their `_score` columns. Whichever field drives `doc-006`'s win becomes obvious.

```esql
-- title-only: doc-006 should dominate (its title literally IS "breaking changes", boosted 3×)
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(title, "8.18 breaking changes", {"boost": 3.0})
| SORT _score DESC | LIMIT 5 | KEEP id, title, _score
```
```esql
-- body-only: doc-057 should surface (its body carries the rare token "8.18")
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body, "8.18 breaking changes")
| SORT _score DESC | LIMIT 5 | KEEP id, title, _score
```

In [ ]:
Q_TITLE_ONLY = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(title, ?q, {"boost": 3.0})
| SORT _score DESC | LIMIT 5 | KEEP id, title, _score
"""
Q_BODY_ONLY = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(body, ?q)
| SORT _score DESC | LIMIT 5 | KEEP id, title, _score
"""

q = "8.18 breaking changes"
print(f"QUERY: {q!r}\n")
print("TITLE-ONLY match (boosted 3x) — who does the boosted title reward?")
show_esql(esql(Q_TITLE_ONLY, q=q), fields=("id", "title"))
print("\nBODY-ONLY match — who does the rare token '8.18' reward?")
show_esql(esql(Q_BODY_ONLY, q=q), fields=("id", "title"))

## Why BM25 got it wrong — and it's *not* "term frequency"

The two queries make the mechanism plain:

- **Title-only:** `doc-006` ("Elasticsearch breaking changes") wins big. Its title is literally those two query words, and `title` is boosted `^3`. The combined-match BM25 query (title OR body) inherits that dominance.
- **Body-only:** `doc-057` ("8.18 release notes") wins, because `8.18` is a rare, high-IDF token living in its body — but its *title* doesn't contain "breaking changes," so in the combined query it can't overcome `doc-006`'s boosted title.

So BM25 picked the doc that matched the **common, boosted words** over the doc that matched the **rare, specific token the user actually cared about**. This is a *field-boost* effect — **not** a term-frequency trap. (Flip the lesson: an aggressive `title^3` boost is a great default that quietly backfires on version-specific queries.)

Notice semantic got this one **right** (`doc-057` at #1) — it understood intent. The two retrievers fail on *opposite* query shapes. That's the whole argument for hybrid.

## (Experimental) `SCORE()` as a one-query alternative

ES|QL has a `SCORE()` function for relevance scores. *If* your cluster build lets `SCORE()` wrap an individual `MATCH` expression, you can show both per-field contributions as columns in a **single** query instead of two:

```esql
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(title, "8.18 breaking changes", {"boost": 3.0}) OR MATCH(body, "8.18 breaking changes")
| EVAL title_score = SCORE(MATCH(title, "8.18 breaking changes", {"boost": 3.0})),
       body_score  = SCORE(MATCH(body,  "8.18 breaking changes"))
| SORT _score DESC | LIMIT 5
| KEEP id, title, _score, title_score, body_score
```

The cell below tries it. **This may error** on builds where `SCORE()` is zero-arg (returns the row's fused score, can't wrap an expression) — that's expected. The two-query approach above is the reliable teaching path; treat this as a "nice if it works."

In [ ]:
Q_SCORE = """
FROM aiewf-workshop-docs METADATA _score
| WHERE MATCH(title, ?q, {"boost": 3.0}) OR MATCH(body, ?q)
| EVAL title_score = SCORE(MATCH(title, ?q, {"boost": 3.0})),
       body_score  = SCORE(MATCH(body,  ?q))
| SORT _score DESC | LIMIT 5
| KEEP id, title, _score, title_score, body_score
"""
try:
    show_esql(esql(Q_SCORE, q="8.18 breaking changes"),
              fields=("id", "title", "_score", "title_score", "body_score"))
except Exception as e:
    print("SCORE(MATCH(...)) not supported on this build — use the two-query approach above.")
    print(f"  ({type(e).__name__}: {str(e)[:160]})")

## Failure mode 3: Paraphrase — BM25 buries the doc it can't lexically match

The classic case *for* semantic search. When a user describes a problem in their own words, the relevant doc often shares almost no vocabulary with the query. BM25 can only score words that overlap — so the right doc sinks, while semantic finds it on meaning.

`doc-049` is about **Watcher**, Elasticsearch's alerting system. It talks about `trigger`, `condition`, `actions`, `schedule`, `webhook` — and does **not** contain the words "notify," "something," or "goes wrong."

A real user types: `notify me when something goes wrong`
- Semantic: finds `doc-049` at/near #1 — that paraphrase *is* the meaning of alerting.
- BM25: `doc-049` is **buried** under docs sharing a stray common word.

In [ ]:
compare("notify me when something goes wrong")
print("\n" + "="*60)
# Same vocabulary gap, second example: a user who wants data tiers but never says "tiers".
compare("reduce storage cost for old logs")  # target doc-041 — semantic #1, BM25 buried

## Why BM25 buries the alerting doc

BM25's score is a sum over **query terms that appear in the doc**. For `notify me when something goes wrong` against `doc-049`, almost none of those words are in the doc — "notify", "something", "goes", "wrong" all contribute ~0. The doc isn't *invisible* (a stray field match keeps it in the results), but its score is tiny and it sinks below docs sharing an incidental common word.

Semantic search has the opposite strength: the query vector and the Watcher doc's vector land in the same region because they *mean* the same thing, with zero shared vocabulary.

> ⚠️ **Buried**, not "scores exactly zero." A real index almost always returns *something*; the failure is that the *right* doc ranks too low to be useful — which, for a user looking at the top 3, is just as broken.

## The core tension — summary table

Every row is what you just ran live against the corpus, in ES|QL.

| Query | Example | Semantic | BM25 (`title^3` OR body) |
|---|---|---|---|
| Exact identifier (blur) | `exit code 137` | ⚠️ #1 but by a hair over a doc with no "137" — unreliable | ✅ decisive #1 (rare token) |
| Bare config value | `new_primaries` | ❌ wrong doc at #1 (plausible neighbor) | ✅ pins the settings doc |
| Distinctive dotted key | `cluster.routing.allocation.enable` | ✅ #1 — sometimes it nails them | ✅ also strong |
| Version-specific | `8.18 breaking changes` | ✅ #1 (understands intent) | ❌ wrong doc #1 (boosted common-word title) |
| Paraphrase / meaning | `notify me when something goes wrong` | ✅ #1 | ❌ buried (no shared vocabulary) |

**The conclusion:** neither retriever is safe alone. Semantic blurs the tokens that must stay exact; BM25 mis-ranks on boosted common words and goes blind to paraphrase. A production system needs **both**, plus a way to fuse their rankings so the retriever that's *right* for a given query wins. In ES|QL that's `FORK ... | FUSE` — Lab 3.

---
*Continue in Discover → Lab 3 assignment, or open `lab3-esql-hybrid-search.ipynb`*